In [3]:
import re
from pathlib import Path

# Nếu True: bỏ qua khi dòng đã có {#...} ở BẤT KỲ đâu
# Nếu False: chỉ bỏ qua khi {#...} nằm ở CUỐI dòng
SKIP_IF_ANCHOR_ANYWHERE = False

# Regex nhận diện anchor {#...} (id có thể là số, chữ, gạch ngang, gạch dưới)
ANCHOR_AT_END_RE = re.compile(r'\{#[^}]+\}\s*$')
ANCHOR_ANYWHERE_RE = re.compile(r'\{#[^}]+\}')


def add_anchor_to_line(line: str) -> str:
    """
    Thêm anchor {#number} vào cuối dòng nếu dòng bắt đầu bằng number\.
    Bỏ qua nếu dòng đã có anchor {#...}.
    Ví dụ: '25\. abc...' -> '25\. abc... {#25}'
    """
    # Tách ký tự xuống dòng
    if line.endswith('\n'):
        content = line[:-1]
        newline = '\n'
    else:
        content = line
        newline = ''

    # === KIỂM TRA ĐÃ CÓ ANCHOR CHƯA ===
    if SKIP_IF_ANCHOR_ANYWHERE:
        if ANCHOR_ANYWHERE_RE.search(content):
            return line
    else:
        if ANCHOR_AT_END_RE.search(content):
            return line

    # Tách phần indent (khoảng trắng đầu dòng)
    stripped = content.lstrip()
    indent = content[:len(content) - len(stripped)]

    # Kiểm tra bắt đầu bằng số
    i = 0
    while i < len(stripped) and stripped[i].isdigit():
        i += 1
    if i == 0:
        return line
    number = stripped[:i]
    rest = stripped[i:]

    # Phải bắt đầu bằng `\.` (backslash + dấu chấm)
    if not rest.startswith('\\.'):
        return line

    # Thêm anchor vào cuối dòng (sau khi xóa khoảng trắng thừa)
    content_stripped = content.rstrip()
    new_content = f"{content_stripped} {{#{number}}}"
    return new_content + newline


def process_file(file_path: str, dry_run: bool = False):
    """Đọc file, thêm anchor cho từng dòng, ghi lại file."""
    path = Path(file_path)
    if not path.exists():
        print(f"❌ File không tồn tại: {file_path}")
        return

    with open(path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    new_lines = [add_anchor_to_line(line) for line in lines]

    # Đếm số dòng thay đổi
    changed = sum(1 for old, new in zip(lines, new_lines) if old != new)

    if dry_run:
        print(f"🔍 [DRY RUN] {file_path}: {changed} dòng sẽ thay đổi")
        # In ra các dòng thay đổi để kiểm tra
        for old, new in zip(lines, new_lines):
            if old != new:
                print(f"   - {old.rstrip()}")
                print(f"   + {new.rstrip()}")
        return

    with open(path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)

    print(f"✅ Đã xử lý: {file_path} ({changed} dòng thay đổi)")


# ====== CẤU HÌNH ======
md_files = [
    "/Users/ng/projects/nikaya2/docs/kinhtrungbo/pali-vi/mn-038-mahatanhasankhayasutta.md",
  
    # Thêm các file khác...
]

# Đặt True để xem trước các thay đổi mà KHÔNG ghi file
DRY_RUN = False

for file in md_files:
    process_file(file, dry_run=DRY_RUN)

print("Hoàn tất!")

✅ Đã xử lý: /Users/ng/projects/nikaya2/docs/kinhtrungbo/pali-vi/mn-038-mahatanhasankhayasutta.md (41 dòng thay đổi)
Hoàn tất!


<>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/0r/x3qsqbx96053svmhzm1mvb_00000gp/T/ipykernel_48246/338082545.py:15: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  Thêm anchor {#number} vào cuối dòng nếu dòng bắt đầu bằng number\.
